In [0]:
import json

dbutils.widgets.text("raw_path", "")
dbutils.widgets.text("adls_account", "3dt2ndteam5cwe")
dbutils.widgets.text("secret_scope", "")
dbutils.widgets.text("secret_key", "")

raw_path = dbutils.widgets.get("raw_path").strip()
adls_account = dbutils.widgets.get("adls_account").strip()
secret_scope = dbutils.widgets.get("secret_scope").strip()
secret_key = dbutils.widgets.get("secret_key").strip()

if not raw_path:
    raise ValueError("raw_path is required")

# raw_path가 abfss://... 이면 계정키 인증 세팅
if raw_path.startswith("abfss://"):
    if not secret_scope or not secret_key:
        raise ValueError("secret_scope and secret_key are required for abfss path")

    account_key = dbutils.secrets.get(secret_scope, secret_key).strip()

    # 잘못된 시크릿(연결문자열) 차단
    if "DefaultEndpointsProtocol=" in account_key or ";" in account_key:
        raise ValueError("secret must be ADLS account key only, not connection string")

    host = f"{adls_account}.dfs.core.windows.net"

    # 충돌 가능 설정 제거
    for k in [
        f"fs.azure.account.auth.type.{host}",
        f"fs.azure.account.oauth.provider.type.{host}",
        f"fs.azure.account.oauth2.client.id.{host}",
        f"fs.azure.account.oauth2.client.secret.{host}",
        f"fs.azure.account.oauth2.client.endpoint.{host}",
        "fs.zazure.account.key",  # 오타 설정 남아있을 때 대비
    ]:
        try:
            spark.conf.unset(k)
        except:
            pass

    spark.conf.set(f"fs.azure.account.key.{host}", account_key)

df = spark.read.json(raw_path)
